<a href="https://colab.research.google.com/github/saravianelson/-E-commerce-Churn-Model-S03-26-Equipo-40-Data-Science/blob/main/S03_Equipo40_Churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Predicción de Churn en E-commerce

**Proyecto:** Simulación de Análisis Predictivo

**Dataset:** Online Retail II


# Entendimiento del Negocio

# El Problema: La fuga de clientes

En un sector tan dinámico como lo es el E-commerce, el crecimiento no depende unicamente de la captación de nuevos clientes/usuarios, sino fundamentalmente de la capacidad de retener los actuales. La tasa de abandono es la métrica crítica que mide cuántos clientes dejan de interactuar con la plataforma en un período determinado de tiempo.

**¿Porqué es vital éste análisis?**

**Costo de adquisición:** Adquirir un nuevo cliente siempre es entre un 5% y un 25% más caro que retener uno existente.

**Rentabilidad:** Un aumento en la retención puede incrementar las ganancias más de un 25%, ya que los clientes recurrentes tienden a comprar con mayor frecuencia elevan el tiket promedio.

**Accionabilidad:** Predecir el Churn nos permite pasar de una postura reactiva a una **proactiva**, identificando clientes en riesgo para intervenir con campañas de marketing(descuentos, programas de lealtad, atención personalizada)

**Objetivos del Proyecto:**

**1. Definir el Churn:** Establecer un criterio técnico de inactividad (90) días, basado en el comportamiento histórico.

**2. Identificar Patrones:** Realizar un EDA para hallar señales, como una baja en la frecuencia de compra, muchas devoluciones, caída del consumo.

**3. Modelado Predictivo:** Entrenar un algoritmo de clasificación que asigne una probabilidad de fuga en cada cliente.

**4. Estrategia de retención:** Segmentar a los clientes según su nivel de riesgo para optimizar el marketing.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")

print("Path to dataset files:", path)

100%|██████████| 14.5M/14.5M [00:00<00:00, 122MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mashlyn/online-retail-ii-uci/versions/3


In [ ]:
import pandas as pd
import os
import datetime as dt


In [ ]:
# 1. Carga de Datos ( Online Retail II - UK )

file_path = os.path.join(path, 'online_retail_II.csv')
df = pd.read_csv(file_path)

# Convert 'InvoiceDate' to datetime objects to enable date calculations
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
# 2. Limpieza Inicial: Solo clientes identificados y sin cancelaciones

df = df[df['Customer ID'].notnull()]
df = df[~df['Invoice'].str.contains('C', na=False)] # 'C' indica cancelación

In [ ]:
# 3. Calculo del total de cada línea

df['TotalSum'] = df['Price'] * df['Quantity']

In [ ]:
# 4. Definir la 'Fecha de Hoy' para el análisis (un día después de la última transacción del dataset)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

In [ ]:
# 5. Creación de la Tabla Maestra (RFM + Churn)
# Agrupamos por Cliente
master_df = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'Invoice': 'nunique',                                  # Frequency
    'TotalSum': 'sum'                                      # Monetary
})

# Renombrar columnas para claridad
master_df.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

In [ ]:
# 6. DEFINICIÓN DE CHURN (Etiqueta para el Modelo)
# Si no ha comprado en más de 90 días, le ponemos etiqueta 1 (Churn)
master_df['CHURN'] = master_df['Recency'].apply(lambda x: 1 if x > 90 else 0)

print(master_df.head())

             Recency  Frequency  Monetary  CHURN
Customer ID                                     
12346.0          326         12  77556.46      1
12347.0            2          8   5633.32      0
12348.0           75          5   2019.40      0
12349.0           19          4   4428.69      0
12350.0          310          1    334.40      1


In [ ]:
# 7. Visualizacion de datos

from google.colab import data_table
data_table.DataTable(master_df, include_index=True, num_rows_per_page=10)

,Recency,Frequency,Monetary,CHURN
Customer ID,,,,
12346.0,326,12,77556.46,1
12347.0,2,8,5633.32,0
12348.0,75,5,2019.40,0
12349.0,19,4,4428.69,0
12350.0,310,1,334.40,1
...,...,...,...,...
18283.0,4,22,2736.65,0
18284.0,432,1,461.68,1
18285.0,661,1,427.00,1
